# V3 (KCDP) Prompt Ablation — Full Metrics (incl. over-tagging diagnostics)

For each condition (baseline, no_rules, reduced, no_kc), metrics are computed against the two human raters (HA, HB) over the **same common pid list** as the main V3 eval, then averaged across the two raters (the AvgHuman view, matching `evaluate_llm_vs_humans`).

Reuses the existing scorer (`problem_f1`, `problem_jaccard`, `evaluate_pair`, `KC_COLUMNS`) without changing its definitions. Precision/recall use the **same empty-set conventions** as `problem_f1`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np

ROOT = Path("/mnt/d/Projects/kintsugi")
sys.path.insert(0, str(ROOT))

from utils.metrics import problem_f1, problem_jaccard, evaluate_pair, KC_COLUMNS

STUDENT_IDS = [10155, 9948, 14189, 14352, 14362, 14363, 14374, 14414, 14474, 14499]
CONDITIONS = ["baseline", "no_rules", "reduced", "no_kc"]

HUMAN_DIR = ROOT / "dataset" / "Rater_KC_Tags" / "Rated_KC_V3"
LLM_DIR = ROOT / "results" / "human_validation" / "ablation"
OUT_PATH = ROOT / "ablation_metrics_full.md"

VALID_KCS = set(KC_COLUMNS)
print("KCs:", len(VALID_KCS), "| Conditions:", CONDITIONS)

KCs: 18 | Conditions: ['baseline', 'no_rules', 'reduced', 'no_kc']


## Loaders (verbatim conventions from `exp21_v3_ablation.ipynb`)

pid key = `{sid}_{pid}`; common set = `human_a ∩ human_b ∩ baseline LLM run`.

In [2]:
def find_one(pattern, directory):
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matched {pattern} in {directory}")
    return matches[-1]

def normalize_gaps(value):
    if isinstance(value, dict):
        gaps = value.get("gaps", [])
    elif isinstance(value, list):
        gaps = value
    else:
        gaps = []
    if not isinstance(gaps, list):
        return set()
    return {g for g in gaps if g in VALID_KCS}

def load_annotation_file(path):
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    sid = str(data.get("studentId", data.get("student_id", "unknown")))
    return sid, {f"{sid}_{pid}": normalize_gaps(v) for pid, v in data.get("annotations", {}).items()}

def merge_files(file_map):
    merged = {}
    for expected_sid, path in file_map.items():
        loaded_sid, anns = load_annotation_file(path)
        if loaded_sid != expected_sid:
            raise ValueError(f"Expected {expected_sid}, found {loaded_sid} in {path.name}")
        merged.update(anns)
    return merged

def load_condition(condition):
    return merge_files({str(sid): LLM_DIR / f"llm_ablation_{condition}_{sid}.json" for sid in STUDENT_IDS})

human_a = merge_files({str(sid): find_one(f"kc_annotations_Pranay Ghuge_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})
human_b = merge_files({str(sid): find_one(f"kc_annotations_Arundhati Das_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})

def common_items():
    """Common set: human_a ∩ human_b ∩ baseline LLM run (same as V3 eval)."""
    ref = load_condition("baseline")
    return sorted(set(human_a) & set(human_b) & set(ref),
                  key=lambda x: (int(x.split("_")[0]), int(x.split("_")[1])))

common = common_items()
print(f"Common problem annotations: {len(common)}")

Common problem annotations: 372


## Per-problem precision / recall — same empty-set conventions as `problem_f1`

In `problem_f1(set_a, set_b)`: `precision = tp/len(set_b)`, `recall = tp/len(set_a)`. Here human is the reference (`set_a`) and LLM is the prediction (`set_b`), so `precision = tp/len(LLM set)`, `recall = tp/len(human set)`. Both-empty = 1.0, exactly-one-empty = 0.0, tp==0 = 0.0.

In [3]:
def problem_precision(human_set, llm_set):
    if not human_set and not llm_set:
        return 1.0
    if not human_set or not llm_set:
        return 0.0
    tp = len(human_set & llm_set)
    if tp == 0:
        return 0.0
    return tp / len(llm_set)

def problem_recall(human_set, llm_set):
    if not human_set and not llm_set:
        return 1.0
    if not human_set or not llm_set:
        return 0.0
    tp = len(human_set & llm_set)
    if tp == 0:
        return 0.0
    return tp / len(human_set)

def mean_pr_vs_human(human, llm, pids):
    """Per-problem mean precision and recall for one human rater vs the LLM."""
    precs, recs = [], []
    for pid in pids:
        h = human.get(pid, set())
        l = llm.get(pid, set())
        precs.append(problem_precision(h, l))
        recs.append(problem_recall(h, l))
    return float(np.mean(precs)), float(np.mean(recs))

## Score each condition (AvgHuman view)

In [4]:
def score_condition(condition, common):
    llm = load_condition(condition)
    missing = [k for k in common if k not in llm]
    if missing:
        raise ValueError(f"{condition} missing {len(missing)} items, e.g. {missing[:3]}")

    # Over-tagging diagnostics (depend only on the LLM gap sets).
    sizes = [len(llm[pid]) for pid in common]
    gap_sizes = [s for s in sizes if s >= 1]
    mean_tags = float(np.mean(sizes))
    mean_tags_gap = float(np.mean(gap_sizes)) if gap_sizes else 0.0

    # Existing scorer: HA-LLM, HB-LLM, then AvgHuman.
    ha = evaluate_pair(human_a, llm, common, KC_COLUMNS)
    hb = evaluate_pair(human_b, llm, common, KC_COLUMNS)
    problem_f1_avg = (ha["Problem_F1"] + hb["Problem_F1"]) / 2
    jaccard_avg = (ha["Jaccard"] + hb["Jaccard"]) / 2
    kappa_avg = (ha["Cohen_kappa"] + hb["Cohen_kappa"]) / 2
    ac1_avg = (ha["Gwet_AC1"] + hb["Gwet_AC1"]) / 2

    # Precision/recall: per-problem mean per rater, then averaged across raters.
    pa, ra = mean_pr_vs_human(human_a, llm, common)
    pb, rb = mean_pr_vs_human(human_b, llm, common)
    mean_precision = (pa + pb) / 2
    mean_recall = (ra + rb) / 2

    return {
        "Condition": condition,
        "mean_tags_per_problem": mean_tags,
        "mean_tags_per_gap_prob": mean_tags_gap,
        "Problem_F1": problem_f1_avg,
        "mean_precision": mean_precision,
        "mean_recall": mean_recall,
        "Jaccard": jaccard_avg,
        "Cohen_kappa": kappa_avg,
        "Gwet_AC1": ac1_avg,
    }

rows = [score_condition(cond, common) for cond in CONDITIONS]

## Build, print, and save the markdown table

In [5]:
cols = [
    "Condition", "mean_tags_per_problem", "mean_tags_per_gap_prob",
    "Problem_F1", "mean_precision", "mean_recall", "Jaccard",
    "Cohen_kappa", "Gwet_AC1",
]

def fmt(c, v):
    return v if c == "Condition" else f"{v:.3f}"

lines = [
    "| " + " | ".join(cols) + " |",
    "|" + "|".join("---" for _ in cols) + "|",
]
for r in rows:
    lines.append("| " + " | ".join(fmt(c, r[c]) for c in cols) + " |")
table = "\n".join(lines)

header = (
    "# V3 (KCDP) Prompt Ablation — Full Metrics\n\n"
    f"AvgHuman view (mean of HA-LLM and HB-LLM) over {len(common)} common "
    "problem annotations (human_a ∩ human_b ∩ baseline LLM run), the same pid "
    "list as the main V3 eval. Problem_F1, Jaccard, Cohen_kappa and Gwet_AC1 "
    "come from `utils.metrics`; precision/recall use the same empty-set "
    "conventions as `problem_f1` (both-empty = 1.0, exactly-one-empty = 0.0), "
    "averaged per-problem then across the two raters. mean_tags columns are "
    "over-tagging diagnostics on the LLM gap sets only.\n\n"
)
OUT_PATH.write_text(header + table + "\n")

print(f"Scored over {len(common)} common problem annotations.\n")
print(table)
print(f"\nWrote {OUT_PATH}")

Scored over 372 common problem annotations.

| Condition | mean_tags_per_problem | mean_tags_per_gap_prob | Problem_F1 | mean_precision | mean_recall | Jaccard | Cohen_kappa | Gwet_AC1 |
|---|---|---|---|---|---|---|---|---|
| baseline | 0.847 | 2.520 | 0.831 | 0.858 | 0.829 | 0.794 | 0.548 | 0.953 |
| no_rules | 0.976 | 2.771 | 0.847 | 0.868 | 0.852 | 0.809 | 0.571 | 0.952 |
| reduced | 0.911 | 2.734 | 0.829 | 0.852 | 0.831 | 0.792 | 0.533 | 0.949 |
| no_kc | 0.745 | 2.131 | 0.820 | 0.859 | 0.809 | 0.782 | 0.496 | 0.950 |

Wrote /mnt/d/Projects/kintsugi/ablation_metrics_full.md
